# imports

In [45]:
import os
import sys
import shutil
import subprocess
import time
from datetime import datetime

# methods

In [46]:
def get_dir_size(start_path):
    total_size = 0
    total_files = 0

    for dirpath, dirnames, filenames in os.walk(start_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            # skip if it is symbolic link
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)
                total_files += 1

    return round(total_size/10**6, 2), total_files


In [41]:
def generate_backup_dir(start_path, backup_dir, root_dir, 
                         skip_dirs, large_dirs, 
                         filetypes_code, filetypes_other, 
                         max_filesize, mod_time_diff, overflow_fcnt,
                         verbose, dry_run):
    
    cnt_files = 0
    cnt_copied = 0
    cnt_skipped = 0
    
    size_total = 0
    size_copied = 0
    size_skipped = 0
    
    
    # saving tree
    ftree = os.path.join(backup_dir, os.path.basename(start_path) + "_tree.html")
    
    # only see upto 1st level to reduce tree file size
    # --du won't work properly as it isn't seeing all the files
    if os.path.basename(start_path) in large_dirs:
        
        ftree_dir_version = os.path.join(backup_dir, os.path.basename(start_path) + "_tree-dir.html")
        
        # save dirs
        subprocess.call(["tree", "-hdDFC", "--du", "--dirsfirst", "-H", start_path, "-o", ftree_dir_version, start_path])
        
        # save files by ignoring some patterns
        subprocess.call(["tree", "-hDFC", "-I", "*txt|*phrases|*story|*lex_rank|*tweet|*json|*html", "--du", "--dirsfirst", "-H", start_path, "-o", ftree, start_path])
    
    
    else:
        subprocess.call(["tree", "-hDFC", "--du", "--dirsfirst", "-H", start_path, "-o", ftree, start_path])
    
    
    if verbose: 
        print("  \nBase Dir:", start_path)
        print("  Backup Dir:", backup_dir)
    
    print("len skip_dirs:", len(skip_dirs))
    
    for base_dir, subdirs, filenames in os.walk(start_path, topdown=True):
        print("  Scanning:", base_dir)
         
        # skip hidden directory 
        # os.path.join(base_dir, d) not in skip_dirs to handle full path 
        # d not in skip_dirs to handle direct ignore: ".ipynb_checkpoints", ".data"
        subdirs[:] = [d for d in subdirs if d[0] != "." and os.path.join(base_dir, d) not in skip_dirs and d not in skip_dirs] 
            
        # splitext('.html') --> ('.html', '') :hidden file, ignore it
        # splitext('1.html') --> ('1', '.html')
        # splitext('2.3.html') --> ('2.3', '.html')
        filenames = [f for f in filenames if os.path.splitext(f)[1] in filetypes_code + filetypes_other]
        cnt_files += len(filenames)
        
        
        if len(filenames) > overflow_fcnt:
            print("\t*****Skipping dir coz of overflow_fcnt:", len(filenames), "*****\n")
            continue
            
            
        if verbose:
            print("\tlen subdirs:", len(subdirs))
            print("\tsubdirs:", [os.path.join(base_dir, d) for d in subdirs[:5]])
            print("\tlen files:", len(filenames))
        
        # tree.txt
        for f in filenames:
            # /home/vivek.a/courses/DeepLearning.ai/tree.txt
            fsrc = os.path.join(base_dir, f)
            
            # skip if it is symbolic link
            if not os.path.islink(fsrc):
                
                # /home/vivek.a/courses/DeepLearning.ai/
                fsrc_dir = os.path.dirname(fsrc)

                # /home/vivek.a/research-backup/courses/DeepLearning.ai/
                # don't use lstrip or strip as it's char based and will remove extra chars
                fdest_dir = os.path.join(backup_dir, fsrc_dir.split(root_dir)[1])
                
                if not os.path.isdir(fdest_dir):
                    os.makedirs(fdest_dir)
                    if verbose: print("\tCreated folder:", fdest_dir)
                
                # /home/vivek.a/research-backup/courses/DeepLearning.ai/tree.txt
                fdest = os.path.join(fdest_dir, f)
                
                fsize = os.path.getsize(fsrc)
                size_total += fsize
                
                # MAX_SIZE is in MB
                # apply MAX_SIZE limit only on filetypes_other
                if os.path.splitext(f)[1] in filetypes_code or (os.path.splitext(f)[1] in filetypes_other and fsize/10**6 <= max_filesize):
                    # file isn't already present in dest_dir: copy it
                    if not os.path.isfile(fdest):
                            if not dry_run: shutil.copy2(fsrc, fdest_dir) 
                            cnt_copied += 1
                            size_copied += fsize
                            if verbose: print("\tCopied first:", fdest)

                    # if file is present: we check modified date difference
                    else:
                        fsrc_time = os.path.getmtime(fsrc)
                        fdest_time = os.path.getmtime(fdest)

                        # if src file is older than mod_time_diff: copy it
                        if fsrc_time - fdest_time > mod_time_diff:
                            if not dry_run: shutil.copy2(fsrc, fdest_dir) 
                            cnt_copied += 1
                            size_copied += fsize
                            if verbose: print("\tCopied mod:", fdest)
                                
                        else:
                            if verbose: print(f"\tSkipped due to time: {fsrc}|  {time.ctime(os.path.getmtime(fsrc)),time.ctime(os.path.getmtime(fsrc))}")
                            
                            cnt_skipped += 1
                            size_skipped += fsize
                
                else:
                    if verbose: print(f"\tSkipped due to size: {fsrc} | size: {fsize}")
                    cnt_skipped += 1
                    size_skipped += fsize
        
        if verbose: print("-"*40, "\n")
    
    print(f"\nTotal files: {cnt_files} |copied: {cnt_copied} |skipped: {cnt_skipped}")
    print(f"Total size: {round(size_total/10**6,2)} MB |copied: {round(size_copied/10**6, 2)} MB |skipped: {round(size_skipped/10**6, 2)} MB")
    print("*"*60, "\n")
    
    return [cnt_files, cnt_copied, cnt_skipped, size_total, size_copied, size_skipped]
          

# configs

In [33]:

user = "vivek.a"
backup_dir = f"/home/{user}/research-backup/"
root_dir = f"/home/{user}/"

skip_dirs = [ ".git", 
               ".ipynb_checkpoints", 
               ".data", 
               "__pycache__",
                
                f"/home/{user}/isb/data_business",
                f"/home/{user}/isb/data_sample",
                f"/home/{user}/isb/data_sp_russell",
                f"/home/{user}/isb/data_sp_russell2",  
                f"/home/{user}/isb/data_yearly",
                
                
                f"/home/{user}/isb/10k_scrape/data_old-full",
                f"/home/{user}/isb/10k_scrape/data_sp-100",
                
                f"/home/{user}/isb/summary/data_cnn_dm",
                f"/home/{user}/isb/summary/op_10K-s&p500",
                f"/home/{user}/isb/summary/op_bertSum",
                f"/home/{user}/isb/summary/data_newsroom",
    
                f"/home/{user}/isb/summary/doc2tweet/op_final_tweets_json",
                f"/home/{user}/isb/summary/doc2tweet/data_news",
                
                f"/home/{user}/kp_extraction/data_paper_abstracts/kp20k/base/keyphrase",
                f"/home/{user}/kp_extraction/data_paper_abstracts/kp20k/base/text_processed",
                f"/home/{user}/kp_extraction/data_paper_abstracts/kp20k/base/text",
    
                f"/home/{user}/kp_extraction/baselines/data/gold_meta-qw",
                f"/home/{user}/kp_extraction/baselines/data/input_processed",
                f"/home/{user}/kp_extraction/baselines/data/input",
                f"/home/{user}/kp_extraction/baselines/data/raw",
                
                f"/home/{user}/kp_extraction/baselines/op_EmbedRank",
                f"/home/{user}/kp_extraction/baselines/op_TextRank",
    
                f"/home/{user}/tech_companies",
    
                f"/home/{user}/Embeddings/CoreNLP-full-2018-02-27",
            ]

large_dirs = ["isb", "kp_extraction", "others"]

folders_to_backup = ['git_repos',
                     'courses',
                     'HASOC-2019',
                     'datasets',
                     'Domain-Indentification',
                     'kp_extraction',
                     'isb',
                     'temp',
                     'nltk_data',
                     'ML',
                     'college',
                     'Embeddings',
                     'others']


# use this to check if there is no unwanted dir  --> as it doesn't have .txt
filetypes_code = [".py", ".ipynb", ".c", ".cpp", ".sh"]

# can have .csv, .pkl as well here since it has limit on max_filesize
filetypes_other = [".txt"]

MAX_FILE_SIZE = 10 # 10 MB
MOD_TIME_DIFF = 1*60*60 # 1 hr

OVERFLOW_FCNT = 100


# run backup

In [38]:

def run_backup(folders_to_backup, backup_dir, root_dir,
            skip_dirs = [".git", ".ipynb_checkpoints", ".data"],
            large_dirs = ["isb", "kp_extraction", "others"],
            filetypes_code = [".py", ".ipynb", ".c", ".cpp", ".sh", ".js"],
            filetypes_other = [".txt", ".csv", ".pkl", ".html", ".json"],
            overflow_fcnt = 100, max_filesize = 10, mod_time_diff = 1*60*60, 
            verbose=False, dry_run=False):
    
    stats = [0,0,0,0,0,0]

    for folder in folders_to_backup:
        folder = os.path.join(root_dir, folder)
        print("Backing up:", folder, os.path.basename(folder))

        # [cnt_files, cnt_copied, cnt_skipped, size_total, size_copied, size_skipped]
        result = generate_backup_dir(folder, backup_dir, root_dir, 
                                     skip_dirs, large_dirs, 
                                     filetypes_code, filetypes_other, 
                                     max_filesize, mod_time_diff, overflow_fcnt,
                                     verbose, dry_run)

        # adding the elements of result with stats
        stats = [sum(x) for x in zip(stats, result)]

    print(f"\nBacked up @ {time.strftime('%d/%m/%Y, %H:%M:%S')}")
    print(f"Total files: {stats[0]} |copied: {stats[1]} |skipped: {stats[2]}")
    print(f"Total size: {round(stats[3]/10**6,2)} MB |copied: {round(stats[4]/10**6, 2)} MB |skipped: {round(stats[5]/10**6, 2)} MB\n")

    

    
def get_backup_details(folders_to_backup, backup_dir):
    """
    gives {file, size} details of backuped up directory
    """
    total_files = 0
    total_size = 0

    for folder in [backup_dir] + folders_to_backup:

        # files on backup dir top level
        if folder == backup_dir:
            files = [os.path.join(backup_dir,f) for f in os.listdir(backup_dir) if os.path.isfile(os.path.join(backup_dir,f))]

            cnt_file = len(files)
            size = 0
            for f in files:
                size += os.path.getsize(f)
            size = round(size/10**6,2)

        else:
            folder = os.path.join(backup_dir, folder)
            size, cnt_file = get_dir_size(folder)

        total_size += size
        total_files += cnt_file

        print(f"{folder:<60} |size: {size} MB |files: {cnt_file}")


    print(f"\nBackup details @ {time.strftime('%d/%m/%Y, %H:%M:%S')}")
    print(f"size: {total_size:.2f} MB |total files: {total_files}\n")
    
    

In [48]:
%%time
verbose = False
dry_run = False

run_backup(folders_to_backup, backup_dir, root_dir, 
             skip_dirs=skip_dirs, large_dirs=large_dirs, 
             filetypes_code=filetypes_code, filetypes_other=filetypes_other, 
             max_filesize=MAX_FILE_SIZE, mod_time_diff=MOD_TIME_DIFF, overflow_fcnt=OVERFLOW_FCNT,
             verbose=verbose, dry_run=dry_run)


Backing up: /home/vivek.a/git_repos git_repos
len skip_dirs: 28
  Scanning: /home/vivek.a/git_repos
  Scanning: /home/vivek.a/git_repos/GithubTest
  Scanning: /home/vivek.a/git_repos/Codes
  Scanning: /home/vivek.a/git_repos/Codes/C Programs
  Scanning: /home/vivek.a/git_repos/Codes/Python OOP
  Scanning: /home/vivek.a/git_repos/useful-scripts

Total files: 26 |copied: 1 |skipped: 25
Total size: 0.21 MB |copied: 0.08 MB |skipped: 0.14 MB
************************************************************ 

Backing up: /home/vivek.a/courses courses
len skip_dirs: 28
  Scanning: /home/vivek.a/courses
  Scanning: /home/vivek.a/courses/pytorch
  Scanning: /home/vivek.a/courses/mlCourse.ai
  Scanning: /home/vivek.a/courses/mlCourse.ai/data
  Scanning: /home/vivek.a/courses/mlCourse.ai/Assignements
  Scanning: /home/vivek.a/courses/mlCourse.ai/tutorials
  Scanning: /home/vivek.a/courses/padh.ai
  Scanning: /home/vivek.a/courses/DeepLearning.ai
  Scanning: /home/vivek.a/courses/DeepLearning.ai/5. nl

# backup details

In [49]:
get_backup_details(folders_to_backup, backup_dir)

/home/vivek.a/research-backup/                               |size: 7.59 MB |files: 19
/home/vivek.a/research-backup/git_repos                      |size: 0.28 MB |files: 27
/home/vivek.a/research-backup/courses                        |size: 6.88 MB |files: 114
/home/vivek.a/research-backup/HASOC-2019                     |size: 1.97 MB |files: 23
/home/vivek.a/research-backup/datasets                       |size: 0.02 MB |files: 2
/home/vivek.a/research-backup/Domain-Indentification         |size: 4.86 MB |files: 35
/home/vivek.a/research-backup/kp_extraction                  |size: 11.84 MB |files: 56
/home/vivek.a/research-backup/isb                            |size: 86.36 MB |files: 78
/home/vivek.a/research-backup/temp                           |size: 0.21 MB |files: 3
/home/vivek.a/research-backup/nltk_data                      |size: 0.0 MB |files: 0
/home/vivek.a/research-backup/ML                             |size: 2.99 MB |files: 5
/home/vivek.a/research-backup/college        

In [50]:
!du -h -d 1 /home/vivek.a/research-backup/

32K	/home/vivek.a/research-backup/datasets
380K	/home/vivek.a/research-backup/git_repos
7.0M	/home/vivek.a/research-backup/courses
2.0M	/home/vivek.a/research-backup/HASOC-2019
4.8M	/home/vivek.a/research-backup/Domain-Indentification
12M	/home/vivek.a/research-backup/kp_extraction
83M	/home/vivek.a/research-backup/isb
212K	/home/vivek.a/research-backup/temp
108K	/home/vivek.a/research-backup/.ipynb_checkpoints
2.9M	/home/vivek.a/research-backup/ML
14M	/home/vivek.a/research-backup/college
552K	/home/vivek.a/research-backup/Embeddings
14M	/home/vivek.a/research-backup/others
52M	/home/vivek.a/research-backup/.git
198M	/home/vivek.a/research-backup/


## git clone

In [56]:
pwd

'/home/vivek.a/git_repos/useful-scripts'

In [58]:
!git status

On branch master
Your branch is up-to-date with 'origin/master'.
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git checkout -- <file>..." to discard changes in working directory)

	modified:   backup_crawler.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


In [60]:
!git add . & git status 

On branch master
Your branch is up-to-date with 'origin/master'.
Changes to be committed:
  (use "git reset HEAD <file>..." to unstage)

	modified:   backup_crawler.ipynb

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git checkout -- <file>..." to discard changes in working directory)

	modified:   backup_crawler.ipynb



In [62]:
!git commit -m "updated with backup details method"
!git push 

[master a41024b] updated with backup details method
 1 file changed, 217 insertions(+), 105 deletions(-)
Counting objects: 3, done.
Delta compression using up to 8 threads.
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 2.04 KiB | 0 bytes/s, done.
Total 3 (delta 2), reused 0 (delta 0)
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To git@github.com:vivenkyan/useful-scripts.git
   e6d5617..a41024b  master -> master


In [ ]:
!cd ../backup_dir/

In [55]:
!cd ../../research-backup/ & git status

On branch master
Your branch is up-to-date with 'origin/master'.
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git checkout -- <file>..." to discard changes in working directory)

	modified:   backup_crawler.ipynb

no changes added to commit (use "git add" and/or "git commit -a")
